# Фонетическая разметка

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Делает GPU невидимыми для PyTorch

## Импорты библиотек

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
from tqdm.auto import tqdm
import Levenshtein
import re

from prosodic import *

Парсинг XML...
Парсинг XML...


Обработка предложений: 100%|██████████| 3372/3372 [00:01<00:00, 2090.07it/s]


Загружено 3372 предложений
Извлечение данных для обучения...
Загружено 41470 слов
С паузами: 9893
Фразовые ударения: 9911

Обучение модели...
Баланс классов: {0: 31577, 1: 9893}
Обучение классификатора...
Обучение регрессора...
Классификация (пауза есть/нет):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6315
           1       1.00      1.00      1.00      1979

    accuracy                           1.00      8294
   macro avg       1.00      1.00      1.00      8294
weighted avg       1.00      1.00      1.00      8294

Регрессия (длительность паузы) MAE: 84.13
Парсинг XML...


Обработка предложений: 100%|██████████| 200/200 [00:00<00:00, 2337.02it/s]


## Извлечение признаков для обучения фонетики на основе XMLParser из просодики

In [3]:
class ProsodyEnhancedFeatureExtractor:    
    def __init__(self, xml_parser, pause_predictor=None):
        self.parser = xml_parser
        self.pause_predictor = pause_predictor
        self.features_data = []
        self.targets_data = []
    
    def extract_features_and_targets_from_xml(self, xml_content, use_predicted_prosody=True):
        sentences_data = self.parser.parse_xml(xml_content)
        
        if use_predicted_prosody and self.pause_predictor:
            sentences_data = self._enhance_with_predicted_prosody(sentences_data)
        
        for sentence_idx, sentence_data in enumerate(sentences_data):
            sentence_features, sentence_targets = self._extract_sentence_features_and_targets(
                sentence_data, sentence_idx
            )
            self.features_data.extend(sentence_features)
            self.targets_data.extend(sentence_targets)
        
        return self.features_data, self.targets_data
    
    def _enhance_with_predicted_prosody(self, sentences_data):
        all_words_data = []
        sentence_word_indices = []
        
        for sentence_idx, sentence_data in enumerate(sentences_data):
            for word_data in sentence_data['words']:
                all_words_data.append(word_data)
                sentence_word_indices.append(sentence_idx)
        
        if all_words_data:
            prediction_result = self.pause_predictor.predict(all_words_data)
            
            word_idx = 0
            for sentence_idx, sentence_data in enumerate(sentences_data):
                for word_data in sentence_data['words']:
                    if word_idx < len(prediction_result['words']):
                        predicted_word = prediction_result['words'][word_idx]
                        
                        word_data['phrasal_stress'] = predicted_word['phrasal_stress']
                        word_data['nucleus'] = '2' if predicted_word['phrasal_stress'] else '0'
                        
                        word_data['predicted_pause_len'] = predicted_word['pause_len']
                        word_data['predicted_has_pause'] = predicted_word['pause_len'] > 0
                        word_data['pause_probability'] = predicted_word['pause_probability']
                        
                        word_idx += 1
        
        return sentences_data
    
    def _extract_sentence_features_and_targets(self, sentence_data, sentence_idx):
        sentence_features = []
        sentence_targets = []
        
        words = sentence_data['words']
        elements = sentence_data['elements']
        
        for word_idx, word_data in enumerate(words):
            prev_word = words[word_idx - 1] if word_idx > 0 else None
            next_word = words[word_idx + 1] if word_idx < len(words) - 1 else None
            
            has_pause_before = self._has_pause_before(elements, word_idx, words)
            has_pause_after = self._has_pause_after(word_data, elements, word_idx)
            
            word_features, word_targets = self._extract_word_features_and_targets(
                word_data, prev_word, next_word, has_pause_before, has_pause_after, 
                sentence_idx, word_idx, len(words)
            )
            sentence_features.append(word_features)
            sentence_targets.append(word_targets)
        
        return sentence_features, sentence_targets
    
    def _has_pause_before(self, elements, word_idx, words):
        if word_idx == 0:
            return False
        
        prev_word = words[word_idx - 1]
        if 'predicted_has_pause' in prev_word:
            return prev_word['predicted_has_pause']
        
        word_count = 0
        for element in elements:
            if element['type'] == 'word':
                if word_count == word_idx - 1:
                    next_idx = elements.index(element) + 1
                    if next_idx < len(elements) and elements[next_idx]['type'] == 'pause':
                        return True
                word_count += 1
        return False
    
    def _has_pause_after(self, word_data, elements, word_idx):
        if 'predicted_has_pause' in word_data:
            return word_data['predicted_has_pause']
        
        word_count = 0
        for element in elements:
            if element['type'] == 'word':
                if word_count == word_idx:
                    next_idx = elements.index(element) + 1
                    if next_idx < len(elements) and elements[next_idx]['type'] == 'pause':
                        return True
                word_count += 1
        return False
    
    def _extract_word_features_and_targets(self, word_data, prev_word, next_word, has_pause_before, has_pause_after, sentence_idx, word_idx, total_words_in_sentence):
        word_content = word_data['content']
        
        current_pause_len = word_data.get('predicted_pause_len', 0)
        pause_len_before = prev_word.get('predicted_pause_len', 0) if prev_word else 0
        
        features = {
            'sentence_id': sentence_idx,
            'word_id': word_idx,
            'word_content': word_content,
            
            'current_word': word_content.lower(),
            'current_nucleus': word_data.get('nucleus', '0'),
            'current_pos': word_data.get('subpart_of_speech', '0'),
            'current_phrasal_stress': word_data.get('phrasal_stress', False),
            'current_pause_len': current_pause_len,
            'pause_probability': word_data.get('pause_probability', 0.0),
            
            'prev_letter': word_content[-1] if len(word_content) > 0 and word_idx > 0 else '<NONE>',
            'next_letter': next_word['content'][0].lower() if next_word and next_word['content'] else '<NONE>',
            
            'has_pause_before': has_pause_before,
            'has_pause_after': has_pause_after,
            'pause_len_before': pause_len_before,
            'pause_len_after': current_pause_len,
            
            'prev_word': prev_word['content'].lower() if prev_word and prev_word['content'] else '<START>',
            'next_word': next_word['content'].lower() if next_word and next_word['content'] else '<END>',
            
            'prev_phrasal_stress': prev_word.get('phrasal_stress', False) if prev_word else False,
            'next_phrasal_stress': next_word.get('phrasal_stress', False) if next_word else False,
            
            'prev_pos': prev_word.get('subpart_of_speech', '0') if prev_word else '<START>',
            'next_pos': next_word.get('subpart_of_speech', '0') if next_word else '<END>',
            
            'word_length': len(word_content),
            'stressed_letter': word_data.get('stressed_letter', False),
            'capitalized': word_content[0].isupper() if word_content else False,
            
            'genesys': word_data.get('genesys', '0'),
            'stress_dict': word_data.get('stress_dict', '0'),
            'semantics2': word_data.get('semantics2', '0'),
            'form': word_data.get('form', '0'),
            
            'letter_count': word_data.get('letter_count', 0),
            'allophone_count': word_data.get('allophone_count', 0),
            
            'is_first_word': word_idx == 0,
            'is_last_word': word_idx == total_words_in_sentence - 1,
            'word_position': word_idx,
            'words_in_sentence': total_words_in_sentence,
        }
        
        targets = self._extract_allophones_sequence(word_data)
        
        return features, targets
    
    def _extract_allophones_sequence(self, word_data):

        if 'allophones' in word_data and word_data['allophones']:
            return word_data['allophones']
        else:
            print(f"Предупреждение: нет аллофонов для слова '{word_data.get('content', '')}'")
            return []
    
    def get_dataframes(self):
        """Возвращает данные в виде DataFrame для признаков и целевых переменных"""
        features_df = pd.DataFrame(self.features_data)
        
        targets_data = []
        for i, (features, targets) in enumerate(zip(self.features_data, self.targets_data)):
            targets_data.append({
                'sentence_id': features['sentence_id'],
                'word_id': features['word_id'],
                'word_content': features['word_content'],
                'allophone_sequence': targets,
                'sequence_length': len(targets)
            })
        
        targets_df = pd.DataFrame(targets_data)
        return features_df, targets_df
    
    def get_training_data(self):
        """Возвращает матрицы X и Y для обучения"""
        features_df, targets_df = self.get_dataframes()
        
        # Признаки для обучения (X)
        feature_columns = [
            'current_word', 'current_nucleus', 'current_pos', 'current_phrasal_stress',
            'prev_letter', 'next_letter', 'has_pause_before', 'has_pause_after',
            'prev_word', 'next_word', 'prev_phrasal_stress', 'next_phrasal_stress',
            'prev_pos', 'next_pos', 'word_length', 'stressed_letter', 'capitalized',
            'current_pause_len', 'pause_len_before', 'pause_len_after', 'pause_probability',
            'genesys', 'stress_dict', 'semantics2', 'form', 'letter_count', 'allophone_count',
            'is_first_word', 'is_last_word', 'word_position', 'words_in_sentence'
        ]
        
        available_columns = [col for col in feature_columns if col in features_df.columns]
        missing_columns = [col for col in feature_columns if col not in features_df.columns]
        
        if missing_columns:
            print(f"Предупреждение: следующие колонки отсутствуют и будут пропущены: {missing_columns}")
        
        X = features_df[available_columns]
        y = targets_df['allophone_sequence']
        
        return X, y

def create_prosody_enhanced_pipeline(xml_file_path, pause_predictor=None, use_predicted_prosody=True):
    with open(xml_file_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()
    
    parser = XMLParser()
    extractor = ProsodyEnhancedFeatureExtractor(parser, pause_predictor)
    
    features, targets = extractor.extract_features_and_targets_from_xml(
        xml_content, use_predicted_prosody=use_predicted_prosody
    )
    
    features_df, targets_df = extractor.get_dataframes()
    
    total_allophones = sum(len(seq) for seq in targets)
    print(f"Извлечено {len(features)} слов")
    print(f"Извлечено {total_allophones} аллофонов")
    print(f"Средняя длина последовательности: {total_allophones / len(features) if features else 0:.2f}")
    
    return features_df, targets_df, extractor

def check_allophones_extraction(xml_file_path):
    with open(xml_file_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()
    
    parser = XMLParser()
    sentences_data = parser.parse_xml(xml_content)
    
    all_allophones = []
    words_with_allophones = 0
    
    for sentence_data in sentences_data:
        for word_data in sentence_data['words']:
            if 'allophones' in word_data and word_data['allophones']:
                words_with_allophones += 1
                all_allophones.extend(word_data['allophones'])
    
    print(f"Слов с аллофонами: {words_with_allophones}")
    print(f"Всего аллофонов: {len(all_allophones)}")
    print(f"Уникальных аллофонов: {len(set(all_allophones))}")
    print(f"Примеры аллофонов: {list(set(all_allophones))[:10]}")
    
    return all_allophones

## Подготовка тренировочных и тестовых дынных

In [ ]:
predictor, xml_parser, extractor = train()

Парсинг XML...
Парсинг XML...


Обработка предложений: 100%|██████████| 3372/3372 [00:01<00:00, 1756.59it/s]


Загружено 3372 предложений
Извлечение данных для обучения...
Загружено 41470 слов
С паузами: 9893
Фразовые ударения: 9911

Обучение модели...
Баланс классов: {0: 31577, 1: 9893}
Обучение классификатора...
Обучение регрессора...
Классификация (пауза есть/нет):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6315
           1       1.00      1.00      1.00      1979

    accuracy                           1.00      8294
   macro avg       1.00      1.00      1.00      8294
weighted avg       1.00      1.00      1.00      8294

Регрессия (длительность паузы) MAE: 84.13


In [5]:
features_df, targets_df, extractor = create_prosody_enhanced_pipeline(
        '/home/danya/datasets/text_to_speech/TEST.xml',
        pause_predictor=predictor,
        use_predicted_prosody=True
    )

X_test, y_test = extractor.get_training_data()

Парсинг XML...


Обработка предложений: 100%|██████████| 2699/2699 [00:01<00:00, 2374.78it/s]


Предупреждение: новое значение в признаке stress_dict, используем '0'
Извлечено 38565 слов
Извлечено 212850 аллофонов
Средняя длина последовательности: 5.52


In [6]:
X_test, y_test

(      current_word current_nucleus current_pos  current_phrasal_stress  \
 0          реклама               0           1                   False   
 1             бога               0           1                   False   
 2         огненной               0           3                   False   
 3           библии               2           1                    True   
 4                в               0           9                   False   
 ...            ...             ...         ...                     ...   
 38560   напоминает               0           6                   False   
 38561    мяукающую               0           7                   False   
 38562         речь               0           1                   False   
 38563      жителей               0           2                   False   
 38564       фикоро               2           1                    True   
 
       prev_letter next_letter  has_pause_before  has_pause_after   prev_word  \
 0          <NONE

In [7]:
# features_df, targets_df, extractor = create_prosody_enhanced_pipeline(
#         '/home/danya/datasets/text_to_speech/crime_and_punishment.Result.xml',
#         pause_predictor=predictor,
#         use_predicted_prosody=True
#     )

features_df, targets_df, extractor = create_prosody_enhanced_pipeline(
        '/home/danya/datasets/text_to_speech/merged.xml',
        pause_predictor=predictor,
        use_predicted_prosody=True
    )
    
X, y = extractor.get_training_data()
    
print(f"Размерность X: {X.shape}")
print(f"Размерность y: {y.shape}")
    
print("\nПризнаки (X):")
print(X.columns.tolist())
    
print("\nПримеры данных:")
print("\nПервые 3 слова:")
for i in range(min(3, len(X))):
    print(f"Слово: {X.iloc[i]['current_word']}")
    print(f"Аллофоны: {y.iloc[i]}")
    print(f"Ударность: {X.iloc[i]['current_nucleus']}")
    print(f"Часть речи: {X.iloc[i]['current_pos']}")
    print(f"Пауза после: {X.iloc[i]['has_pause_after']}")
    print(f"Предыдущая буква: {X.iloc[i]['prev_letter']}")
    print(f"Следующая буква: {X.iloc[i]['next_letter']}")
    print("---")

Парсинг XML...


Обработка предложений: 100%|██████████| 8505/8505 [00:05<00:00, 1571.24it/s]


Предупреждение: новое значение в признаке stress_dict, используем '0'
Извлечено 122476 слов
Извлечено 640760 аллофонов
Средняя длина последовательности: 5.23
Размерность X: (122476, 31)
Размерность y: (122476,)

Признаки (X):
['current_word', 'current_nucleus', 'current_pos', 'current_phrasal_stress', 'prev_letter', 'next_letter', 'has_pause_before', 'has_pause_after', 'prev_word', 'next_word', 'prev_phrasal_stress', 'next_phrasal_stress', 'prev_pos', 'next_pos', 'word_length', 'stressed_letter', 'capitalized', 'current_pause_len', 'pause_len_before', 'pause_len_after', 'pause_probability', 'genesys', 'stress_dict', 'semantics2', 'form', 'letter_count', 'allophone_count', 'is_first_word', 'is_last_word', 'word_position', 'words_in_sentence']

Примеры данных:

Первые 3 слова:
Слово: в
Аллофоны: ['v']
Ударность: 0
Часть речи: 9
Пауза после: False
Предыдущая буква: <NONE>
Следующая буква: н
---
Слово: начале
Аллофоны: ['n', 'a1', 'ch', 'a0', "l'", 'i4']
Ударность: 0
Часть речи: 1
Пауза по

## LSTM модель и подготовка данных

In [8]:
class PhonemeDataset(Dataset):
    def __init__(self, X, y_sequences, char_to_idx, phoneme_to_idx, max_length=20):
        self.X = X.reset_index(drop=True)
        self.y_sequences = y_sequences.reset_index(drop=True)
        self.char_to_idx = char_to_idx
        self.phoneme_to_idx = phoneme_to_idx
        self.max_length = max_length
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        word = str(self.X.iloc[idx]['current_word']).lower()
        sequence = self.y_sequences.iloc[idx]
        
        word_encoded = [self.char_to_idx.get(char, self.char_to_idx['<UNK>']) for char in word]
        
        if isinstance(sequence, list):
            phonemes_encoded = [self.phoneme_to_idx.get(ph, self.phoneme_to_idx['<UNK>']) for ph in sequence]
        else:
            phonemes_encoded = [self.phoneme_to_idx['<UNK>']]
        
        word_padded = self.pad_sequence(word_encoded, self.max_length, self.char_to_idx['<PAD>'])
        phonemes_padded = self.pad_sequence(phonemes_encoded, self.max_length, self.phoneme_to_idx['<PAD>'])
        
        mask = torch.tensor([1 if i < len(word_encoded) else 0 for i in range(self.max_length)], dtype=torch.bool)
        
        return (
            torch.tensor(word_padded, dtype=torch.long),
            torch.tensor(phonemes_padded, dtype=torch.long),
            mask
        )
    
    def pad_sequence(self, sequence, max_length, pad_value):
        if len(sequence) >= max_length:
            return sequence[:max_length]
        else:
            return sequence + [pad_value] * (max_length - len(sequence))

def collate_fn(batch):
    words, phonemes, masks = zip(*batch)
    
    words_tensor = torch.stack(words)
    phonemes_tensor = torch.stack(phonemes)
    masks_tensor = torch.stack(masks)
    
    return words_tensor, phonemes_tensor, masks_tensor

class ContextAwareLSTM(nn.Module):
    def __init__(self, vocab_size, phoneme_vocab_size, embedding_dim=64, hidden_dim=128, 
                 num_layers=2, dropout=0.3, max_length=20, context_dim=15):
        super(ContextAwareLSTM, self).__init__()
        
        self.vocab_size = vocab_size
        self.phoneme_vocab_size = phoneme_vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.max_length = max_length
        
        self.char_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim + context_dim,  # символы + контекстные признаки
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        
        self.output_layer = nn.Linear(hidden_dim * 2, phoneme_vocab_size)  # *2 для bidirectional
        
        self.init_weights()
    
    def init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
        nn.init.xavier_uniform_(self.output_layer.weight)
    
    def forward(self, word_chars, context_features, mask=None):
        # word_chars: [batch_size, seq_len]
        # context_features: [batch_size, seq_len, context_dim]
        # mask: [batch_size, seq_len]
        
        batch_size, seq_len = word_chars.size()
        char_embedded = self.char_embedding(word_chars)  # [batch_size, seq_len, embedding_dim]
        combined = torch.cat([char_embedded, context_features], dim=-1)
        lstm_out, _ = self.lstm(combined)  # [batch_size, seq_len, hidden_dim * 2]
        
        if mask is not None:
            lstm_out = lstm_out * mask.unsqueeze(-1)
        
        lstm_out = self.dropout(lstm_out)
        output = self.output_layer(lstm_out)  # [batch_size, seq_len, phoneme_vocab_size]
        return output

class SimpleLSTMTranscriber(nn.Module):
    def __init__(self, vocab_size, phoneme_vocab_size, embedding_dim=64, hidden_dim=128, 
                 num_layers=2, dropout=0.3):
        super(SimpleLSTMTranscriber, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dim * 2, phoneme_vocab_size)
        
        self.init_weights()
    
    def init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
        nn.init.xavier_uniform_(self.output_layer.weight)
    
    def forward(self, x, mask=None):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = self.dropout(lstm_out)
        
        if mask is not None:
            lstm_out = lstm_out * mask.unsqueeze(-1)
            
        output = self.output_layer(lstm_out)
        return output

class TorchPhoneticTranscriber:
    def __init__(self, max_length=20, device=None):
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.max_length = max_length
        self.model = None
        self.char_to_idx = {}
        self.phoneme_to_idx = {}
        self.idx_to_phoneme = {}
        
    def prepare_vocabularies(self, X, y_sequences):
        all_chars = set()
        for word in X['current_word']:
            if isinstance(word, str):
                all_chars.update(word.lower())
        
        self.char_to_idx = {'<PAD>': 0, '<UNK>': 1}
        for idx, char in enumerate(sorted(all_chars), start=2):
            self.char_to_idx[char] = idx
        
        all_phonemes = set()
        for sequence in y_sequences:
            if isinstance(sequence, list):
                all_phonemes.update(sequence)
        
        self.phoneme_to_idx = {'<PAD>': 0, '<UNK>': 1}
        for idx, phoneme in enumerate(sorted(all_phonemes), start=2):
            self.phoneme_to_idx[phoneme] = idx
        
        self.idx_to_phoneme = {idx: ph for ph, idx in self.phoneme_to_idx.items()}
        
        print(f"Размер словаря символов: {len(self.char_to_idx)}")
        print(f"Размер словаря аллофонов: {len(self.phoneme_to_idx)}")
    
    def prepare_context_features(self, word_tensor):
        batch_size, seq_len = word_tensor.shape
        
        context_features = torch.zeros(batch_size, seq_len, 15, device=self.device)
        
        for i in range(batch_size):
            for j in range(seq_len):
                if word_tensor[i, j] != 0:
                    context_features[i, j, 0] = j / seq_len  # позиция в слове
                    context_features[i, j, 1] = (seq_len - j - 1) / seq_len  # позиция с конца
                    context_features[i, j, 2] = 1.0 if j == 0 else 0.0  # начало слова
                    context_features[i, j, 3] = 1.0 if j == seq_len - 1 else 0.0  # конец слова
        
        return context_features
    
    def create_data_loader(self, X, y_sequences, batch_size=32, shuffle=True):
        """Создает DataLoader"""
        dataset = PhonemeDataset(X, y_sequences, self.char_to_idx, self.phoneme_to_idx, self.max_length)
        return DataLoader(
            dataset, 
            batch_size=batch_size, 
            shuffle=shuffle, 
            num_workers=0,
            collate_fn=collate_fn
        )
    
    def build_model(self, use_context=True):
        vocab_size = len(self.char_to_idx)
        phoneme_vocab_size = len(self.phoneme_to_idx)
        
        if use_context:
            self.model = ContextAwareLSTM(
                vocab_size=vocab_size,
                phoneme_vocab_size=phoneme_vocab_size,
                embedding_dim=64,
                hidden_dim=128,
                num_layers=2,
                dropout=0.3,
                max_length=self.max_length,
                context_dim=15
            )
        else:
            self.model = SimpleLSTMTranscriber(
                vocab_size=vocab_size,
                phoneme_vocab_size=phoneme_vocab_size,
                embedding_dim=64,
                hidden_dim=128,
                num_layers=2,
                dropout=0.3
            )
        
        self.model.to(self.device)
        return self.model
    
    def train_model(self, train_loader, val_loader=None, epochs=50, learning_rate=0.001):
        if self.model is None:
            raise ValueError("Модель не построена. Сначала вызовите build_model()")
            
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        optimizer = optim.Adam(self.model.parameters(), lr=learning_rate, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        
        train_losses = []
        val_losses = []
        train_accuracies = []
        val_accuracies = []
        
        for epoch in range(epochs):
            self.model.train()
            total_loss = 0
            total_correct = 0
            total_tokens = 0
            
            for batch_idx, (word_chars, target_phonemes, mask) in enumerate(train_loader):
                word_chars = word_chars.to(self.device)
                target_phonemes = target_phonemes.to(self.device)
                mask = mask.to(self.device)
                
                optimizer.zero_grad()
                
                if isinstance(self.model, ContextAwareLSTM):
                    context_features = self.prepare_context_features(word_chars)
                    outputs = self.model(word_chars, context_features, mask)
                else:
                    outputs = self.model(word_chars, mask)
                
                loss = criterion(outputs.reshape(-1, outputs.size(-1)), target_phonemes.reshape(-1))
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                
                total_loss += loss.item()
                
                predictions = outputs.argmax(dim=-1)
                non_padding = target_phonemes != 0
                correct = (predictions[non_padding] == target_phonemes[non_padding]).sum().item()
                total_correct += correct
                total_tokens += non_padding.sum().item()
            
            train_loss = total_loss / len(train_loader)
            train_accuracy = total_correct / total_tokens if total_tokens > 0 else 0
            train_losses.append(train_loss)
            train_accuracies.append(train_accuracy)
            
            val_loss = 0
            val_accuracy = 0
            if val_loader:
                self.model.eval()
                total_val_loss = 0
                total_val_correct = 0
                total_val_tokens = 0
                
                with torch.no_grad():
                    for word_chars, target_phonemes, mask in val_loader:
                        word_chars = word_chars.to(self.device)
                        target_phonemes = target_phonemes.to(self.device)
                        mask = mask.to(self.device)
                        
                        if isinstance(self.model, ContextAwareLSTM):
                            context_features = self.prepare_context_features(word_chars)
                            outputs = self.model(word_chars, context_features, mask)
                        else:
                            outputs = self.model(word_chars, mask)
                        
                        loss = criterion(outputs.reshape(-1, outputs.size(-1)), target_phonemes.reshape(-1))
                        total_val_loss += loss.item()
                        
                        predictions = outputs.argmax(dim=-1)
                        non_padding = target_phonemes != 0
                        correct = (predictions[non_padding] == target_phonemes[non_padding]).sum().item()
                        total_val_correct += correct
                        total_val_tokens += non_padding.sum().item()
                
                val_loss = total_val_loss / len(val_loader)
                val_accuracy = total_val_correct / total_val_tokens if total_val_tokens > 0 else 0
                val_losses.append(val_loss)
                val_accuracies.append(val_accuracy)
                
                scheduler.step(val_loss)
            
            if (epoch + 1) % 5 == 0:
                print(f'Epoch {epoch+1}/{epochs}:')
                print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}')
                if val_loader:
                    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}')
        
        return {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accuracies': train_accuracies,
            'val_accuracies': val_accuracies
        }
    
    def predict(self, X):
        if self.model is None:
            raise ValueError("Модель не обучена!")
            
        self.model.eval()
        predictions = []
        
        temp_dataset = PhonemeDataset(
            X, 
            pd.Series([[]] * len(X)),
            self.char_to_idx, 
            self.phoneme_to_idx, 
            self.max_length
        )
        
        temp_loader = DataLoader(
            temp_dataset, 
            batch_size=32, 
            shuffle=False,
            collate_fn=collate_fn
        )
        
        with torch.no_grad():
            for word_chars, _, mask in temp_loader:
                word_chars = word_chars.to(self.device)
                mask = mask.to(self.device)
                
                if isinstance(self.model, ContextAwareLSTM):
                    context_features = self.prepare_context_features(word_chars)
                    outputs = self.model(word_chars, context_features, mask)
                else:
                    outputs = self.model(word_chars, mask)
                
                batch_predictions = outputs.argmax(dim=-1).cpu().numpy()
                
                for i in range(batch_predictions.shape[0]):
                    sequence = []
                    word_mask = mask[i].cpu().numpy()
                    
                    for j in range(batch_predictions.shape[1]):
                        if word_mask[j] and batch_predictions[i, j] != 0:
                            phoneme_idx = batch_predictions[i, j]
                            if phoneme_idx in self.idx_to_phoneme:
                                phoneme = self.idx_to_phoneme[phoneme_idx]
                                if phoneme not in ['<PAD>', '<UNK>']:
                                    sequence.append(phoneme)
                    
                    predictions.append(sequence)
        
        return predictions
    
    def save_model(self, filepath):
        model_data = {
            'model_state_dict': self.model.state_dict(),
            'char_to_idx': self.char_to_idx,
            'phoneme_to_idx': self.phoneme_to_idx,
            'idx_to_phoneme': self.idx_to_phoneme,
            'max_length': self.max_length,
            'model_type': type(self.model).__name__
        }
        torch.save(model_data, filepath)
        print(f"Модель сохранена в {filepath}")
    
    def load_model(self, filepath, use_context=True):
        model_data = torch.load(filepath, map_location=self.device)
        
        self.char_to_idx = model_data['char_to_idx']
        self.phoneme_to_idx = model_data['phoneme_to_idx']
        self.idx_to_phoneme = model_data['idx_to_phoneme']
        self.max_length = model_data['max_length']
        
        self.build_model(use_context=use_context)
        self.model.load_state_dict(model_data['model_state_dict'])
        self.model.to(self.device)
        
        print(f"Модель загружена из {filepath}")

def create_and_train_transcriber(X, y_sequences, use_context=False, epochs=30):
    transcriber = TorchPhoneticTranscriber(max_length=20)
    transcriber.prepare_vocabularies(X, y_sequences)
    train_loader = transcriber.create_data_loader(X, y_sequences, batch_size=32, shuffle=True)
    transcriber.build_model(use_context=use_context)
    history = transcriber.train_model(train_loader, epochs=epochs)
    
    return transcriber, history



## Тест модели

In [9]:
class PhoneticTranscriberTester:
    def __init__(self, transcriber, char_to_idx, phoneme_to_idx, idx_to_phoneme):
        self.transcriber = transcriber
        self.char_to_idx = char_to_idx
        self.phoneme_to_idx = phoneme_to_idx
        self.idx_to_phoneme = idx_to_phoneme
    
    def remove_stress(self, phoneme_sequence):
        """Удаляет ударения из фонем (преобразует 'a2' в 'a')"""
        return [ph[0] if len(ph) > 1 and ph[1].isdigit() else ph for ph in phoneme_sequence]
    
    def calculate_levenshtein_distance(self, true_sequences, pred_sequences):
        """Расстояние Левенштейна, нормированное на длину истинной транскрипции"""
        total_normalized_distance = 0
        total_sequences = len(true_sequences)
        
        for true_seq, pred_seq in zip(true_sequences, pred_sequences):
            distance = Levenshtein.distance(true_seq, pred_seq)
            normalized_distance = distance / len(true_seq) if len(true_seq) > 0 else 1.0
            total_normalized_distance += normalized_distance
        
        avg_normalized_distance = total_normalized_distance / total_sequences if total_sequences > 0 else 1.0
        return avg_normalized_distance
    
    def calculate_phoneme_metrics(self, true_sequences, pred_sequences, with_stress=True):
        """Вычисляет Accuracy, Precision, Recall, F-score для фонем"""
        all_true_phonemes = []
        all_pred_phonemes = []
        
        for true_seq, pred_seq in zip(true_sequences, pred_sequences):
            if not with_stress:
                # Удаляем ударения для фонемного уровня
                true_seq = self.remove_stress(true_seq)
                pred_seq = self.remove_stress(pred_seq)
            
            min_len = min(len(true_seq), len(pred_seq))
            all_true_phonemes.extend(true_seq[:min_len])
            all_pred_phonemes.extend(pred_seq[:min_len])
        
        # Фильтруем служебные токены
        filtered_true = []
        filtered_pred = []
        
        for true_ph, pred_ph in zip(all_true_phonemes, all_pred_phonemes):
            if true_ph not in ['<PAD>', '<UNK>'] and pred_ph not in ['<PAD>', '<UNK>']:
                filtered_true.append(true_ph)
                filtered_pred.append(pred_ph)
        
        if not filtered_true:
            return 0, 0, 0, 0
        
        accuracy = accuracy_score(filtered_true, filtered_pred)
        
        unique_phonemes = list(set(filtered_true + filtered_pred))
        precision = precision_score(filtered_true, filtered_pred, 
                                  labels=unique_phonemes, average='micro', zero_division=0)
        recall = recall_score(filtered_true, filtered_pred, 
                            labels=unique_phonemes, average='micro', zero_division=0)
        f1 = f1_score(filtered_true, filtered_pred, 
                     labels=unique_phonemes, average='micro', zero_division=0)
        
        return accuracy, precision, recall, f1
    
    def calculate_wrr(self, true_sequences, pred_sequences, with_stress=True):
        """
        Word Recognition Rate (WRR)
        Слово считается правильным если ВСЕ фонемы/аллофоны совпадают
        """
        correct_words = 0
        total_words = len(true_sequences)
        
        for true_seq, pred_seq in zip(true_sequences, pred_sequences):
            if not with_stress:
                # Удаляем ударения для фонемного уровня
                true_seq = self.remove_stress(true_seq)
                pred_seq = self.remove_stress(pred_seq)
            
            if true_seq == pred_seq:
                correct_words += 1
        
        wrr = correct_words / total_words if total_words > 0 else 0
        return wrr
    
    def test_model(self, X_test, y_test, batch_size=32):
        print("=== ТЕСТИРОВАНИЕ МОДЕЛИ ===")
        print("Выполнение предсказаний...")
        predictions = self.transcriber.predict(X_test)
        
        true_sequences = []
        for seq in y_test:
            if isinstance(seq, list):
                true_sequences.append(seq)
            else:
                true_sequences.append([])
        
        valid_indices = [i for i, seq in enumerate(true_sequences) if len(seq) > 0]
        filtered_true = [true_sequences[i] for i in valid_indices]
        filtered_pred = [predictions[i] for i in valid_indices]
        
        if not filtered_true:
            print("Нет валидных данных для тестирования!")
            return {}
        
        print(f"Протестировано {len(filtered_true)} слов")
        
        print("Вычисление стандартных метрик для аллофонов (с ударениями)...")
        accuracy_allophones, precision_allophones, recall_allophones, f1_allophones = \
            self.calculate_phoneme_metrics(filtered_true, filtered_pred, with_stress=True)
        
        print("Вычисление стандартных метрик для фонем (без ударений)...")
        accuracy_phonemes, precision_phonemes, recall_phonemes, f1_phonemes = \
            self.calculate_phoneme_metrics(filtered_true, filtered_pred, with_stress=False)
        
        print("Вычисление нормированного расстояния Левенштейна...")
        levenshtein_distance = self.calculate_levenshtein_distance(filtered_true, filtered_pred)
        
        print("Вычисление WRR для аллофонов (с ударениями)...")
        wrr_allophones = self.calculate_wrr(filtered_true, filtered_pred, with_stress=True)
        
        print("Вычисление WRR для фонем (без ударений)...")
        wrr_phonemes = self.calculate_wrr(filtered_true, filtered_pred, with_stress=False)
        
        # Проверка порогов для Online теста
        phonemes_threshold_passed = wrr_phonemes >= 0.71
        allophones_threshold_passed = wrr_allophones >= 0.55
        
        results = {
            # Стандартные метрики для аллофонов (с ударениями)
            'accuracy_allophones': accuracy_allophones,
            'precision_allophones': precision_allophones,
            'recall_allophones': recall_allophones,
            'f1_allophones': f1_allophones,
            
            # Стандартные метрики для фонем (без ударений)
            'accuracy_phonemes': accuracy_phonemes,
            'precision_phonemes': precision_phonemes,
            'recall_phonemes': recall_phonemes,
            'f1_phonemes': f1_phonemes,
            
            # Расстояние Левенштейна
            'levenshtein_distance': levenshtein_distance,
            
            # WRR метрики
            'wrr_allophones': wrr_allophones,
            'wrr_phonemes': wrr_phonemes,
            
            # Пороговые проверки для Online теста
            'phonemes_threshold_passed': phonemes_threshold_passed,
            'allophones_threshold_passed': allophones_threshold_passed,
            'phonemes_threshold': 0.71,
            'allophones_threshold': 0.55,
            
            # Статистика
            'total_tested': len(filtered_true),
            'true_sequences': filtered_true,
            'pred_sequences': filtered_pred
        }
        
        return results
    
    def print_detailed_results(self, results):
        print("\n" + "="*60)
        print("ДЕТАЛИЗИРОВАННЫЕ РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
        print("="*60)
        
        print(f"\nСТАНДАРТНЫЕ МЕТРИКИ - АЛЛОФОНЫ (с ударениями):")
        print(f"  Accuracy:  {results['accuracy_allophones']:.4f}")
        print(f"  Precision: {results['precision_allophones']:.4f}")
        print(f"  Recall:    {results['recall_allophones']:.4f}")
        print(f"  F-score:   {results['f1_allophones']:.4f}")
        
        print(f"\nСТАНДАРТНЫЕ МЕТРИКИ - ФОНЕМЫ (без ударений):")
        print(f"  Accuracy:  {results['accuracy_phonemes']:.4f}")
        print(f"  Precision: {results['precision_phonemes']:.4f}")
        print(f"  Recall:    {results['recall_phonemes']:.4f}")
        print(f"  F-score:   {results['f1_phonemes']:.4f}")
        
        print(f"\nРАССТОЯНИЕ ЛЕВЕНШТЕЙНА:")
        print(f"  Нормированное расстояние: {results['levenshtein_distance']:.4f}")
        
        print(f"\nWORD RECOGNITION RATE (WRR):")
        print(f"  WRR для аллофонов (с ударениями): {results['wrr_allophones']:.4f}")
        print(f"  WRR для фонем (без ударений):     {results['wrr_phonemes']:.4f}")
        
        print(f"\nONLINE ТЕСТ - ПОРОГОВЫЕ ПРОВЕРКИ:")
        phonemes_status = "ПРОЙДЕН" if results['phonemes_threshold_passed'] else "НЕ ПРОЙДЕН"
        allophones_status = "ПРОЙДЕН" if results['allophones_threshold_passed'] else "НЕ ПРОЙДЕН"
        
        print(f"  Фонемы (WRR >= {results['phonemes_threshold']}): {phonemes_status}")
        print(f"  Аллофоны (WRR >= {results['allophones_threshold']}): {allophones_status}")
        
        overall_status = "ПРОЙДЕН" if (results['phonemes_threshold_passed'] and 
                                     results['allophones_threshold_passed']) else "НЕ ПРОЙДЕН"
        print(f"  ОБЩИЙ РЕЗУЛЬТАТ: {overall_status}")
        
        print(f"\nСТАТИСТИКА:")
        print(f"  Всего протестировано слов: {results['total_tested']}")
        print(f"  Правильных слов (фонемы): {int(results['wrr_phonemes'] * results['total_tested'])}")
        print(f"  Правильных слов (аллофоны): {int(results['wrr_allophones'] * results['total_tested'])}")
    
    def print_examples(self, X_test, y_test, num_examples=10):
        print(f"\nПРИМЕРЫ ПРЕДСКАЗАНИЙ (первые {num_examples}):")
        print("-" * 80)
        
        predictions = self.transcriber.predict(X_test.head(num_examples))
        
        for i, (word, true_seq, pred_seq) in enumerate(zip(
            X_test.head(num_examples)['current_word'], 
            y_test.head(num_examples), 
            predictions
        )):
            # Проверяем правильность для обоих уровней
            true_phonemes = self.remove_stress(true_seq)
            pred_phonemes = self.remove_stress(pred_seq)
            
            is_correct_phonemes = true_phonemes == pred_phonemes
            is_correct_allophones = true_seq == pred_seq
            
            status_phonemes = "✓" if is_correct_phonemes else "✗"
            status_allophones = "✓" if is_correct_allophones else "✗"
            
            print(f"{i+1}. Слово: '{word}' [Фонемы:{status_phonemes} Аллофоны:{status_allophones}]")
            print(f"   Истинные аллофоны: {true_seq}")
            print(f"   Предсказанные:     {pred_seq}")
            print(f"   Истинные фонемы:   {true_phonemes}")
            print(f"   Предсказанные:     {pred_phonemes}")
            print()

def run_comprehensive_test(transcriber, X_test, y_test):
    tester = PhoneticTranscriberTester(
        transcriber=transcriber,
        char_to_idx=transcriber.char_to_idx,
        phoneme_to_idx=transcriber.phoneme_to_idx,
        idx_to_phoneme=transcriber.idx_to_phoneme
    )
    
    results = tester.test_model(X_test, y_test)
    
    if results:
        tester.print_detailed_results(results)
        tester.print_examples(X_test, y_test, num_examples=10)
        return results
    else:
        print("Тестирование не удалось - нет валидных данных")
        return None

## Обучение модели

In [14]:
transcriber, history = create_and_train_transcriber(X, y, epochs=100)

Размер словаря символов: 94
Размер словаря аллофонов: 61
Epoch 5/100:
  Train Loss: 0.2152, Train Acc: 0.9311
Epoch 10/100:
  Train Loss: 0.1831, Train Acc: 0.9413
Epoch 15/100:
  Train Loss: 0.1702, Train Acc: 0.9453
Epoch 20/100:
  Train Loss: 0.1640, Train Acc: 0.9473
Epoch 25/100:
  Train Loss: 0.1608, Train Acc: 0.9483
Epoch 30/100:
  Train Loss: 0.1584, Train Acc: 0.9493
Epoch 35/100:
  Train Loss: 0.1570, Train Acc: 0.9500
Epoch 40/100:
  Train Loss: 0.1549, Train Acc: 0.9504
Epoch 45/100:
  Train Loss: 0.1540, Train Acc: 0.9509
Epoch 50/100:
  Train Loss: 0.1525, Train Acc: 0.9512
Epoch 55/100:
  Train Loss: 0.1518, Train Acc: 0.9515
Epoch 60/100:
  Train Loss: 0.1522, Train Acc: 0.9516
Epoch 65/100:
  Train Loss: 0.1501, Train Acc: 0.9519
Epoch 70/100:
  Train Loss: 0.1504, Train Acc: 0.9518
Epoch 75/100:
  Train Loss: 0.1497, Train Acc: 0.9523
Epoch 80/100:
  Train Loss: 0.1497, Train Acc: 0.9522
Epoch 85/100:
  Train Loss: 0.1494, Train Acc: 0.9523
Epoch 90/100:
  Train Loss

## Тестирование модели

In [15]:
results = run_comprehensive_test(transcriber, X_test, y_test)

=== ТЕСТИРОВАНИЕ МОДЕЛИ ===
Выполнение предсказаний...
Протестировано 38565 слов
Вычисление стандартных метрик для аллофонов (с ударениями)...
Вычисление стандартных метрик для фонем (без ударений)...
Вычисление нормированного расстояния Левенштейна...
Вычисление WRR для аллофонов (с ударениями)...
Вычисление WRR для фонем (без ударений)...

ДЕТАЛИЗИРОВАННЫЕ РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ

СТАНДАРТНЫЕ МЕТРИКИ - АЛЛОФОНЫ (с ударениями):
  Accuracy:  0.9778
  Precision: 0.9778
  Recall:    0.9778
  F-score:   0.9778

СТАНДАРТНЫЕ МЕТРИКИ - ФОНЕМЫ (без ударений):
  Accuracy:  0.9849
  Precision: 0.9849
  Recall:    0.9849
  F-score:   0.9849

РАССТОЯНИЕ ЛЕВЕНШТЕЙНА:
  Нормированное расстояние: 0.1748

WORD RECOGNITION RATE (WRR):
  WRR для аллофонов (с ударениями): 0.5358
  WRR для фонем (без ударений):     0.5581

ONLINE ТЕСТ - ПОРОГОВЫЕ ПРОВЕРКИ:
  Фонемы (WRR >= 0.71): НЕ ПРОЙДЕН
  Аллофоны (WRR >= 0.55): НЕ ПРОЙДЕН
  ОБЩИЙ РЕЗУЛЬТАТ: НЕ ПРОЙДЕН

СТАТИСТИКА:
  Всего протестировано слов: 38565


In [12]:
results

{'accuracy_allophones': 0.9049037816011408,
 'precision_allophones': 0.9049037816011408,
 'recall_allophones': 0.9049037816011408,
 'f1_allophones': 0.9049037816011408,
 'accuracy_phonemes': 0.942014142414011,
 'precision_phonemes': 0.942014142414011,
 'recall_phonemes': 0.942014142414011,
 'f1_phonemes': 0.942014142414011,
 'levenshtein_distance': 0.22533634643527325,
 'wrr_allophones': 0.4531310774017892,
 'wrr_phonemes': 0.5038506417736289,
 'phonemes_threshold_passed': False,
 'allophones_threshold_passed': False,
 'phonemes_threshold': 0.71,
 'allophones_threshold': 0.55,
 'total_tested': 38565,
 'true_sequences': [["r'", 'i1', 'k', 'l', 'a0', 'm', 'a4'],
  ['b', 'o0', 'g', 'a4'],
  ['o0', 'g', "n'", 'i4', 'n', 'a4', 'j'],
  ["b'", 'i0', 'b', "l'", 'i4', 'i4'],
  ['v'],
  ['o0', 'g', "n'", 'i4', 'n', 'a4', 'm'],
  ["m'", 'i0', "r'", 'i4'],
  ['s', 'v', 'a0', 'r', "g'", 'i4'],
  ['n', 'a2'],
  ['z', 'a2', 'l', 'a1', 't', 'o0', 'm'],
  ['p', 'u1', "t'", 'i0'],
  ['d', 'a0', 'a4'],
 

## Предсказание на реальных данных

In [13]:
features_df, targets_df, extractor = create_prosody_enhanced_pipeline(
        '/home/danya/Downloads/Telegram Desktop/Example_Input_Phonetics.xml',
        pause_predictor=predictor,
        use_predicted_prosody=True
    )

X_real, y_real = extractor.get_training_data()

Парсинг XML...


Обработка предложений: 100%|██████████| 1/1 [00:00<00:00, 1227.84it/s]

Предупреждение: нет аллофонов для слова 'Больше'
Предупреждение: нет аллофонов для слова 'всего'
Предупреждение: нет аллофонов для слова 'человек'
Предупреждение: нет аллофонов для слова 'ненавидит'
Предупреждение: нет аллофонов для слова 'тех,'
Предупреждение: нет аллофонов для слова 'кто'
Предупреждение: нет аллофонов для слова 'смеется'
Предупреждение: нет аллофонов для слова 'ему'
Предупреждение: нет аллофонов для слова 'прямо'
Предупреждение: нет аллофонов для слова 'в'
Предупреждение: нет аллофонов для слова 'глаза,'
Предупреждение: нет аллофонов для слова 'а'
Предупреждение: нет аллофонов для слова 'не'
Предупреждение: нет аллофонов для слова 'тех,'
Предупреждение: нет аллофонов для слова 'кто'
Предупреждение: нет аллофонов для слова 'смеется'
Предупреждение: нет аллофонов для слова 'за'
Предупреждение: нет аллофонов для слова 'его'
Предупреждение: нет аллофонов для слова 'спиной.'
Извлечено 19 слов
Извлечено 0 аллофонов
Средняя длина последовательности: 0.00


In [16]:
def predict_real_data(transcriber, X_real, output_json_path):
    predictions = transcriber.predict(X_real)
    
    output_data = {
        "words": []
    }
    
    for i, (word, pred_sequence) in enumerate(zip(X_real['current_word'], predictions)):
        word_data = {
            "content": word,
            "allophones": pred_sequence
        }
        output_data["words"].append(word_data)
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump([output_data], f, ensure_ascii=False, indent=4)
    
    print(f"Предсказания сохранены в: {output_json_path}")
    print(f"Обработано слов: {len(predictions)}")
    
    return output_data

def predict_real_data_with_sentences(transcriber, X_real, output_json_path, sentence_length=15):  
    predictions = transcriber.predict(X_real)
    
    sentences = []
    current_sentence = {"words": []}
    
    for i, (word, pred_sequence) in enumerate(zip(X_real['current_word'], predictions)):
        word_data = {
            "content": word,
            "allophones": pred_sequence
        }
        current_sentence["words"].append(word_data)
        
        is_end_of_sentence = (
            word in ['.', '!', '?'] or
            len(current_sentence["words"]) >= sentence_length or
            i == len(predictions) - 1
        )
        
        if is_end_of_sentence and current_sentence["words"]:
            sentences.append(current_sentence)
            current_sentence = {"words": []}
    
    sentences = [s for s in sentences if s["words"]]
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(sentences, f, ensure_ascii=False, indent=4)
    
    print(f"Предсказания сохранены в: {output_json_path}")
    print(f"Создано предложений: {len(sentences)}")
    print(f"Обработано слов: {len(predictions)}")
    
    return sentences

if __name__ == "__main__":
    features_real, targets_real, extractor_real = create_prosody_enhanced_pipeline(
        '/home/danya/Downloads/Telegram Desktop/Test_Phonetics.xml',
        pause_predictor=predictor,
        use_predicted_prosody=True
    )
    
    X_real, y_real = extractor_real.get_training_data()
    
    output_data = predict_real_data(
        transcriber, 
        X_real, 
        'Davydov_phonetic_rev2.json'
    )

Парсинг XML...


Обработка предложений: 100%|██████████| 200/200 [00:00<00:00, 2598.78it/s]

Предупреждение: нет аллофонов для слова 'Реально'
Предупреждение: нет аллофонов для слова 'начать'
Предупреждение: нет аллофонов для слова 'день'
Предупреждение: нет аллофонов для слова 'с'
Предупреждение: нет аллофонов для слова 'улыбки -'
Предупреждение: нет аллофонов для слова 'это'
Предупреждение: нет аллофонов для слова 'проснуться'
Предупреждение: нет аллофонов для слова 'от'
Предупреждение: нет аллофонов для слова 'звонка'
Предупреждение: нет аллофонов для слова 'будильника'
Предупреждение: нет аллофонов для слова 'в'
Предупреждение: нет аллофонов для слова 'одну'
Предупреждение: нет аллофонов для слова 'минуту'
Предупреждение: нет аллофонов для слова 'первого'
Предупреждение: нет аллофонов для слова 'ночи,'
Предупреждение: нет аллофонов для слова 'широко'
Предупреждение: нет аллофонов для слова 'улыбнуться'
Предупреждение: нет аллофонов для слова 'и'
Предупреждение: нет аллофонов для слова 'спать'
Предупреждение: нет аллофонов для слова 'дальше.'
Предупреждение: нет аллофонов д

Предсказания сохранены в: Davydov_phonetic_rev2.json
Обработано слов: 3012


---